In [6]:
import pandas as pd
import numpy as np
from sklearn.decomposition import PCA
# ---------- 1. 读取原始数据（获取原始特征） ----------
train_raw = pd.read_csv("train.csv")
test_raw = pd.read_csv("test.csv")

# ---------- 2. 读取正确的聚类标签（来自 K-means聚类模型.py） ----------
train_cluster = pd.read_csv("train_with_cluster_final.csv")
test_cluster = pd.read_csv("test_with_cluster_final.csv")

# 将 Cluster 列合并到原始数据中（确保按行合并，最好用索引）
# 最简单的方式：直接赋值，因为两边的行数一致
train_raw['Cluster_Label'] = train_cluster['Cluster'].values
test_raw['Cluster_Label'] = test_cluster['Cluster'].values

# ---------- 3. 构造衍生特征 ----------
for df in [train_raw, test_raw]:
    df['HouseAge'] = 2010 - df['YearBuilt']
    df['AreaPerRoom'] = df['GrLivArea'] / df['TotRmsAbvGrd']
    
# ---------- 4. 选择要使用的特征 ----------
features = [
    'OverallQual', 'GrLivArea', 'TotalBsmtSF', 'GarageCars', 
    '1stFlrSF', 'FullBath','HalfBath', 'TotRmsAbvGrd', 'YearRemodAdd',
    'KitchenQual', 'BsmtQual','BsmtFinSF1', 'FireplaceQu', 'MSZoning',
    'HouseAge', 'AreaPerRoom', 'Cluster_Label','Neighborhood',
    'LotArea',          # 地块面积
        'MasVnrArea',       # 石材饰面面积
        'BsmtUnfSF',        # 地下室未装修面积
        'WoodDeckSF',       # 木甲板面积
        'OpenPorchSF',      # 开放式门廊面积
        'EnclosedPorch',    # 封闭式门廊面积
        'ScreenPorch',      # 纱窗门廊面积
        'PoolArea',         # 泳池面积
        'MiscVal',          # 杂项价值
]

train_df = train_raw[features].copy()
test_df = test_raw[features].copy()

# ---------- 5. 分别对 train 和 test 进行预处理，避免泄漏 ----------
def preprocess(df):
    df = df.copy()
    # 5a. 缺失值填充（固定值，无泄漏）
    for col in df.columns:
        if col == 'Neighborhood':
            continue
        if df[col].dtype == 'object':
            df[col] = df[col].fillna('None')
        else:
            df[col] = df[col].fillna(0)
    
    # 5b. 质量编码
    quality_map = {'Ex': 5, 'Gd': 4, 'TA': 3, 'Fa': 2, 'Po': 1, 'None': 0}
    for col in ['KitchenQual', 'BsmtQual', 'FireplaceQu','ExterQual']:
        if col in df.columns:
            df[col] = df[col].map(quality_map)
    
    # 5c. 对数化（独立变换，无泄漏）
    for col in ['GrLivArea', 'TotalBsmtSF', '1stFlrSF', 'AreaPerRoom','PoolArea','MiscVal',
               'LotArea','EnclosedPorch','ScreenPorch','OpenPorchSF', 'MasVnrArea','WoodDeckSF',
              'BsmtFinSF1','BsmtUnfSF' ]:
        df[col] = np.log1p(df[col])
    
    return df

train_processed = preprocess(train_df)
test_processed = preprocess(test_df)

# ---------- 6. 对类别特征进行编码（重要：用 train 确定编码，再应用到 test） ----------
# 合并仅为了一次性 fit 编码器（这里用 pd.get_dummies 但分别进行更安全）
# 更简单的做法：合并编码，因为类别空间固定，但为了严谨，分开做：
train_processed = pd.get_dummies(train_processed, columns=['MSZoning','Neighborhood'], drop_first=True)
test_processed = pd.get_dummies(test_processed, columns=['MSZoning','Neighborhood'], drop_first=True)



# 对齐列：确保 test 的列与 train 一致（缺少的补0，多余的删掉）
train_cols = train_processed.columns
test_processed = test_processed.reindex(columns=train_cols, fill_value=0)


# ---------- 7. 保存 ----------
train_processed['SalePrice'] = train_raw['SalePrice'].values
train_processed.to_csv("train_final.csv", index=False)
test_processed.to_csv("test_final.csv", index=False)

In [7]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import xgboost as xgb
import warnings
from sklearn.ensemble import RandomForestRegressor
warnings.filterwarnings('ignore')

# 中文显示
plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False

# ==================== 1. 加载数据 ====================

train_df = pd.read_csv("train_final.csv")
X_train = train_df.drop('SalePrice', axis=1)
y_train_raw = train_df['SalePrice'].values
y_train = np.log1p(y_train_raw)
X_test = pd.read_csv("test_final.csv")

print(f"训练特征: {X_train.shape}, 测试特征: {X_test.shape}")
print(f"房价范围: {y_train_raw.min()} ~ {y_train_raw.max()}")

# ==================== 2. 训练/验证拆分 ====================
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train, y_train, test_size=0.2, random_state=42
)

# ==================== 3. XGBoost 模型训练（简化参数） ====================
from xgboost.callback import EarlyStopping



model = xgb.XGBRegressor(
    n_estimators=400,
    max_depth=3,
    learning_rate=0.05,
    subsample=0.7,
    colsample_bytree=0.7,
    reg_alpha=0.2,
    reg_lambda=1.0,
   
    
    
    random_state=42,
    eval_metric='rmse',
    callbacks=[EarlyStopping(rounds=25, save_best=True)]   # ✅ 移到构造函数中
)

model.fit(
    X_tr, y_tr,
    eval_set=[(X_val, y_val)],
 
    verbose=False
)

print(f"✅ 训练完成，最优迭代次数：{model.best_iteration}")
print("\n" + "="*60)
print("【随机森林模型】")
print("="*60)

rf_model = RandomForestRegressor(
    n_estimators=400,      # 树的数量
    max_depth=15,          # 最大深度（可调）
    min_samples_split=5,   # 内部节点再划分所需最小样本数
    min_samples_leaf=2,    # 叶子节点最少样本数
    random_state=42,
    n_jobs=-1              # 使用所有CPU核心
)

# 训练
rf_model.fit(X_tr, y_tr)
# 预测（自动使用最佳迭代数）
y_tr_pred = model.predict(X_tr)
y_val_pred = model.predict(X_val)
y_tr_pred_rf = rf_model.predict(X_tr)
y_val_pred_rf = rf_model.predict(X_val)
# 转回原始价格
y_tr_real = np.expm1(y_tr)
y_val_real = np.expm1(y_val)
y_tr_pred_real = np.expm1(y_tr_pred)
y_val_pred_real = np.expm1(y_val_pred)
y_tr_pred_real_rf = np.expm1(y_tr_pred_rf)
y_val_pred_real_rf = np.expm1(y_val_pred_rf)
def metrics(y_true, y_pred):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    return {"RMSE":rmse, "MAE":mae, "R2":r2}
# 随机森林评估
m_tr_rf = metrics(y_tr, y_tr_pred_rf)
m_val_rf = metrics(y_val, y_val_pred_rf)
m_tr_real_rf = metrics(y_tr_real, y_tr_pred_real_rf)
m_val_real_rf = metrics(y_val_real, y_val_pred_real_rf)

print(f"训练集(log)  RMSE: {m_tr_rf['RMSE']:.6f}  R²: {m_tr_rf['R2']:.4f}")
print(f"验证集(log)  RMSE: {m_val_rf['RMSE']:.6f}  R²: {m_val_rf['R2']:.4f}")
print(f"\n训练集(原始) RMSE: ${m_tr_real_rf['RMSE']:.2f}  R²: {m_tr_real_rf['R2']:.4f}")
print(f"验证集(原始) RMSE: ${m_val_real_rf['RMSE']:.2f}  R²: {m_val_real_rf['R2']:.4f}")

# 随机森林残差统计
resid_rf = y_val_real - y_val_pred_real_rf
print(f"\n残差均值: {resid_rf.mean():.2f}")
print(f"残差标准差: {resid_rf.std():.2f}")
print(f"平均相对误差: {np.mean(np.abs(resid_rf)/y_val_real)*100:.2f}%")

print("\n" + "="*60)
print("【K-means+xgboost模型】")
print("="*60)

m_tr = metrics(y_tr, y_tr_pred)
m_val = metrics(y_val, y_val_pred)
print(f"训练集(log)  RMSE: {m_tr['RMSE']:.6f}  R²: {m_tr['R2']:.4f}")
print(f"验证集(log)  RMSE: {m_val['RMSE']:.6f}  R²: {m_val['R2']:.4f}")

m_tr_real = metrics(y_tr_real, y_tr_pred_real)
m_val_real = metrics(y_val_real, y_val_pred_real)
print(f"\n训练集(原始) RMSE: ${m_tr_real['RMSE']:.2f}  R²: {m_tr_real['R2']:.4f}")
print(f"验证集(原始) RMSE: ${m_val_real['RMSE']:.2f}  R²: {m_val_real['R2']:.4f}")
print("\n" + "="*60)
m_tr_rf = metrics(y_tr, y_tr_pred_rf)
m_val_rf = metrics(y_val, y_val_pred_rf)
m_tr_real_rf = metrics(y_tr_real, y_tr_pred_real_rf)
m_val_real_rf = metrics(y_val_real, y_val_pred_real_rf)
# ==================== 5. 残差统计 ====================
resid = y_val_real - y_val_pred_real
print("\n" + "="*60)
print("【残差统计】")
print("="*60)
print(f"残差均值: {resid.mean():.2f}")
print(f"残差标准差: {resid.std():.2f}")
print(f"平均相对误差: {np.mean(np.abs(resid)/y_val_real)*100:.2f}%")

# ==================== 6. 特征重要性图 ====================

imp = pd.DataFrame({
    "Feature": X_train.columns,
    "Importance": model.feature_importances_
}).sort_values("Importance", ascending=False)

plt.figure(figsize=(10,6))
plt.barh(imp['Feature'][:10], imp['Importance'][:10])
plt.xlabel("重要性")
plt.title("XGBoost Top10 特征重要性")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig("xgb_feature_importance.png", dpi=300)
plt.close()

print("\n特征重要性图已保存：xgb_feature_importance.png")
print("\nTop5 重要特征：")
print(imp.head(10).to_string(index=False))

# ==================== 7. 生成 Kaggle 提交文件 ====================
test_pred_log = model.predict(X_test)
test_pred = np.expm1(test_pred_log)

test_raw = pd.read_csv("test.csv")
submission = pd.DataFrame({
    "Id": test_raw["Id"],
    "SalePrice": test_pred
})
submission.to_csv("kaggle_submission_xgb.csv", index=False)

# ==================== 8. 总结报告 ====================
print("\n" + "="*60)
print("【模型验证总结报告】")
print("="*60)
print(f"验证集 R²: {m_val['R2']:.4f}")
print(f"验证集 RMSE: ${m_val_real['RMSE']:.2f}")



训练特征: (1460, 53), 测试特征: (1459, 53)
房价范围: 34900 ~ 755000
✅ 训练完成，最优迭代次数：240

【随机森林模型】
训练集(log)  RMSE: 0.072399  R²: 0.9656
验证集(log)  RMSE: 0.151601  R²: 0.8768

训练集(原始) RMSE: $14991.26  R²: 0.9623
验证集(原始) RMSE: $30539.74  R²: 0.8784

残差均值: 3842.62
残差标准差: 30297.03
平均相对误差: 10.66%

【K-means+xgboost模型】
训练集(log)  RMSE: 0.092506  R²: 0.9439
验证集(log)  RMSE: 0.137652  R²: 0.8985

训练集(原始) RMSE: $17113.27  R²: 0.9509
验证集(原始) RMSE: $28831.51  R²: 0.8916


【残差统计】
残差均值: 2287.47
残差标准差: 28740.62
平均相对误差: 9.70%

特征重要性图已保存：xgb_feature_importance.png

Top5 重要特征：
    Feature  Importance
OverallQual    0.229119
KitchenQual    0.174942
 GarageCars    0.084288
   BsmtQual    0.059195
FireplaceQu    0.055910
  GrLivArea    0.055632
   HouseAge    0.047820
MSZoning_RL    0.030809
TotalBsmtSF    0.025548
   FullBath    0.021771

【模型验证总结报告】
验证集 R²: 0.8985
验证集 RMSE: $28831.51
